In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="10-personalized-news-feed",
    notebook_name="01_news_feed_system_design.ipynb"
)

# Personalized News Feed: System Design

## The Big Idea (For a 12-Year-Old)

Imagine you open Instagram or TikTok after school. There are **thousands** of new posts from your friends, pages you follow, and groups you are in. But your phone screen can only show a few posts at a time. How does the app decide which posts to show you *first*?

That is what a **personalized news feed** does. It is like a super-smart friend who pre-reads everything and says: "Hey, I think you will love *these* posts." Instead of showing you posts in the order they were created (which would bury the good stuff), it **ranks** every post by how much YOU specifically would enjoy it.

Every major platform does this:
- **Facebook** ranks posts from friends, groups, and pages based on your past behavior
- **Twitter/X** mixes chronological tweets with "top tweets" it thinks you will like
- **LinkedIn** prioritizes professional content based on your career interests

---

## Staff-Level Technical Summary

We design a **Facebook-style personalized news feed** using:
- **Multi-task learning** with shared backbone + N task-specific heads (predict likes, comments, shares, hides, dwell time, skip)
- **Weighted engagement score** combining predicted probabilities across reaction types
- **Pointwise Learning-to-Rank** via binary classification per reaction
- **User-author affinity features** as the single strongest predictor
- **Three-stage serving**: Retrieval -> Ranking -> Re-ranking (under 200ms)
- **Offline metrics**: ROC-AUC per task | **Online metrics**: CTR, reaction rates, time spent, user satisfaction surveys

## 1. Problem Definition and Clarifying Requirements

### The Interview Starts Here

In a real interview, you must clarify before building. Think of it like a builder asking the homeowner questions before starting construction -- you need to know what you are building before you pick up the tools.

In [ ]:
# Let's organize the clarifying requirements as structured data
# This is how you'd think about it systematically in an interview

requirements = {
    "business_objective": "Keep users engaged so they stay on the platform longer (increases ad revenue)",
    "platform_scope": "Social media news feed (Facebook-style)",
    "new_activities": "Unseen posts AND posts with unseen comments",
    "content_types": ["text", "images", "videos", "combinations"],
    "engagement_types": {
        "explicit_positive": ["like", "comment", "share", "friend_request"],
        "implicit_positive": ["click", "dwell_time"],
        "negative": ["hide", "block", "report", "skip"]
    },
    "latency_requirement": "Ranked posts displayed within 200ms",
    "scale": {
        "total_users": "~3 billion",
        "daily_active_users": "~2 billion",
        "feed_checks_per_day": "~2 per user"
    }
}

print("=== Personalized News Feed Requirements ===")
print(f"\nBusiness Objective: {requirements['business_objective']}")
print(f"Latency: {requirements['latency_requirement']}")
print(f"Scale: {requirements['scale']['daily_active_users']} DAU, {requirements['scale']['feed_checks_per_day']} checks/day")
print(f"\nContent Types: {', '.join(requirements['content_types'])}")
print(f"\nEngagement Types:")
for category, types in requirements['engagement_types'].items():
    print(f"  {category}: {', '.join(types)}")

print("\n=== Key Insight ===")
print("The core problem: Given a user, return a ranked list of posts ordered")
print("by how engaging each post is to THAT SPECIFIC user.")
print("Do this in under 200ms for 2 billion daily active users.")

### Why Revenue Depends on the News Feed

This is worth understanding deeply because it drives every design decision:

```
Better News Feed  -->  Users see engaging content
                  -->  Users stay on platform longer
                  -->  More ad impressions
                  -->  More revenue
```

Facebook generates almost **all** of its revenue from ads inserted between news feed posts. So even a 0.1% improvement in feed quality translates to millions of dollars.

## 2. Framing as an ML Task

### What Should We Optimize?

Think of it like running a school cafeteria. You want to predict which foods each student will enjoy most. You have three options:

| Option | Analogy | Pros | Cons |
|--------|---------|------|------|
| **Maximize clicks/dwell** | Track which foods kids look at | Tons of data | Looking is not the same as liking (clickbait!) |
| **Maximize likes/shares** | Track which foods kids rave about | Strong signal | Very few students actually tell you they liked it |
| **Weighted combination** | Combine both signals | Best of both worlds | Need to choose weights carefully |

**We choose Option 3: Weighted engagement score.** This lets the business tune what matters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === Engagement Score: The Heart of the News Feed ===

# Each reaction type gets a weight based on business value
# Think of weights like 'how much do we care about this reaction?'

reaction_weights = {
    'click': 1,           # Weak signal -- everyone clicks
    'like': 5,            # Decent signal -- user explicitly approved
    'comment': 10,        # Strong signal -- user invested time to write
    'share': 20,          # Very strong -- user vouches for this to friends
    'friend_request': 30, # Strongest -- led to a new connection
    'hide': -10,          # Negative -- user explicitly said 'I don't want this'
    'block': -50,         # Very negative -- user wants nothing from this author
}

# Visualize the weights
fig, ax = plt.subplots(figsize=(10, 5))
reactions = list(reaction_weights.keys())
weights = list(reaction_weights.values())
colors = ['#2ecc71' if w > 0 else '#e74c3c' for w in weights]

bars = ax.barh(reactions, weights, color=colors, edgecolor='white', linewidth=1.5)
ax.set_xlabel('Weight', fontsize=12)
ax.set_title('Reaction Weights in Engagement Score', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.5)

# Add value labels
for bar, weight in zip(bars, weights):
    x_pos = bar.get_width() + 1 if weight > 0 else bar.get_width() - 4
    ax.text(x_pos, bar.get_y() + bar.get_height()/2, f'{weight:+d}',
            va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("Why these weights?")
print("  - A share is worth 20x a click because it brings NEW eyeballs to the platform")
print("  - A block is worth -50 because it means we showed something truly bad")
print("  - A hide is milder (-10) -- maybe the user just isn't interested right now")

In [ ]:
# === How the Engagement Score Works ===

# For each (user, post) pair, we predict P(reaction) for each reaction type.
# Then: Engagement Score = sum(P(reaction_i) * weight_i)

def compute_engagement_score(predicted_probs, weights):
    """
    Compute engagement score for a single (user, post) pair.
    
    12-year-old version:
    If our model says there's a 48% chance you'll like this post,
    a 12% chance you'll comment, and a 4% chance you'll share it,
    we multiply each by how much we 'value' that reaction and add them up.
    """
    score = 0.0
    details = []
    for reaction, prob in predicted_probs.items():
        weight = weights.get(reaction, 0)
        contribution = prob * weight
        score += contribution
        details.append((reaction, prob, weight, contribution))
    return score, details


# Example: Model predictions for one (user, post) pair
example_predictions = {
    'click': 0.23,
    'like': 0.48,
    'comment': 0.12,
    'share': 0.04,
    'friend_request': 0.001,
    'hide': 0.02,
    'block': 0.001,
}

score, details = compute_engagement_score(example_predictions, reaction_weights)

print("=== Engagement Score Calculation ===")
print(f"{'Reaction':<18} {'P(reaction)':<14} {'Weight':<10} {'Score':<10}")
print("-" * 52)
for reaction, prob, weight, contribution in details:
    print(f"{reaction:<18} {prob:<14.3f} {weight:<10d} {contribution:<10.3f}")
print("-" * 52)
print(f"{'TOTAL':<18} {'':<14} {'':<10} {score:<10.3f}")

print(f"\nThis post gets an engagement score of {score:.3f}")
print("Posts are ranked by this score -- higher = shown first.")

### ML Category: Pointwise Learning to Rank

We use **multiple binary classifiers** (one per reaction type) to predict the probability that a user will perform each reaction on each post. This is called **pointwise** LTR because we score each post independently.

```
Input:   (user_features, post_features) for one (user, post) pair
Output:  P(click), P(like), P(comment), P(share), P(hide), P(block), ...
Ranking: Sort all posts by weighted engagement score
```

**Why pointwise instead of pairwise or listwise?**
- At Facebook scale (billions of posts), pairwise comparisons (O(n^2)) are too expensive
- Pointwise scoring is O(n) -- score each post independently, then sort
- Multi-task learning lets us share computation across reaction predictions

## 3. Training Data

### Where Does the Data Come From?

Think of it like a school keeping records. Every time a student sees a cafeteria menu item (impression) and decides to eat it (like) or skip it (no reaction), that gets logged.

In [ ]:
import pandas as pd
from datetime import datetime, timedelta

# === Raw Data Sources ===
# In production, these are massive tables stored in data warehouses

np.random.seed(42)

# 1. Users Table
n_users = 20
users_df = pd.DataFrame({
    'user_id': range(1, n_users + 1),
    'username': [f'user_{i}' for i in range(1, n_users + 1)],
    'age': np.random.randint(18, 60, n_users),
    'gender': np.random.choice(['M', 'F', 'Other'], n_users),
    'country': np.random.choice(['US', 'UK', 'India', 'Brazil'], n_users, p=[0.4, 0.2, 0.2, 0.2]),
})

# 2. Posts Table
n_posts = 50
post_types = ['text', 'image', 'video', 'text+image']
posts_df = pd.DataFrame({
    'post_id': range(1, n_posts + 1),
    'author_id': np.random.randint(1, n_users + 1, n_posts),
    'post_type': np.random.choice(post_types, n_posts, p=[0.3, 0.35, 0.2, 0.15]),
    'text_content': [f'Post about {topic}' for topic in np.random.choice(
        ['sports', 'food', 'travel', 'tech', 'music', 'pets', 'politics', 'memes'],
        n_posts)],
    'num_likes': np.random.exponential(50, n_posts).astype(int),
    'num_comments': np.random.exponential(10, n_posts).astype(int),
    'num_shares': np.random.exponential(5, n_posts).astype(int),
    'created_at': [datetime(2024, 3, 15) - timedelta(hours=np.random.exponential(48))
                   for _ in range(n_posts)],
})

# 3. Friendships Table (bidirectional)
n_friendships = 60
friendships = set()
while len(friendships) < n_friendships:
    u1, u2 = np.random.randint(1, n_users + 1, 2)
    if u1 != u2:
        friendships.add((min(u1, u2), max(u1, u2)))
friendships_df = pd.DataFrame(list(friendships), columns=['user_id_1', 'user_id_2'])
friendships_df['close_friend'] = np.random.choice([0, 1], len(friendships_df), p=[0.8, 0.2])

# 4. Interactions Table (the gold mine!)
n_interactions = 2000
interaction_types = ['impression', 'click', 'like', 'comment', 'share', 'hide', 'skip']
interaction_probs = [0.50, 0.20, 0.12, 0.06, 0.03, 0.02, 0.07]
interactions_df = pd.DataFrame({
    'user_id': np.random.randint(1, n_users + 1, n_interactions),
    'post_id': np.random.randint(1, n_posts + 1, n_interactions),
    'interaction_type': np.random.choice(interaction_types, n_interactions, p=interaction_probs),
    'dwell_time_sec': np.random.exponential(5, n_interactions).round(1),
})

print("=== Data Tables ===")
print(f"\nUsers: {len(users_df)} rows")
print(users_df.head(5).to_string(index=False))
print(f"\nPosts: {len(posts_df)} rows")
print(posts_df[['post_id', 'author_id', 'post_type', 'text_content', 'num_likes']].head(5).to_string(index=False))
print(f"\nFriendships: {len(friendships_df)} rows")
print(friendships_df.head(5).to_string(index=False))
print(f"\nInteractions: {len(interactions_df)} rows")
print(interactions_df.head(5).to_string(index=False))

In [ ]:
# === Constructing Training Data for Multi-Task Learning ===

def construct_multitask_training_data(interactions_df):
    """
    Build training dataset for each binary classification task.
    
    12-year-old version:
    For each post that was shown to a user, we ask MULTIPLE questions:
    - Did they click it?  (yes=1, no=0)
    - Did they like it?   (yes=1, no=0)
    - Did they comment?   (yes=1, no=0)
    - Did they share it?  (yes=1, no=0)
    - Did they hide it?   (yes=1, no=0)
    
    Staff-level detail:
    - Positive: user performed the reaction on the post
    - Negative: user saw the post (impression) but did NOT perform the reaction
    - Each task has its own class imbalance (shares are much rarer than clicks)
    """
    # Get all unique (user, post) pairs that had impressions
    impressions = interactions_df[interactions_df['interaction_type'] == 'impression'][
        ['user_id', 'post_id']].drop_duplicates()
    
    tasks = ['click', 'like', 'comment', 'share', 'hide']
    task_data = {}
    
    for task in tasks:
        # Get positive pairs for this task
        positives = interactions_df[interactions_df['interaction_type'] == task][
            ['user_id', 'post_id']].drop_duplicates()
        positives[f'label_{task}'] = 1
        
        # Merge with impressions to get labels
        merged = impressions.merge(positives, on=['user_id', 'post_id'], how='left')
        merged[f'label_{task}'] = merged[f'label_{task}'].fillna(0).astype(int)
        
        task_data[task] = merged
        
        pos_count = merged[f'label_{task}'].sum()
        neg_count = len(merged) - pos_count
        print(f"Task '{task}': {pos_count} positive ({pos_count/len(merged)*100:.1f}%), "
              f"{neg_count} negative ({neg_count/len(merged)*100:.1f}%)")
    
    # Combine all task labels into one DataFrame
    combined = impressions.copy()
    for task in tasks:
        combined = combined.merge(
            task_data[task][['user_id', 'post_id', f'label_{task}']],
            on=['user_id', 'post_id'], how='left'
        )
    
    return combined


print("=== Multi-Task Training Data Construction ===")
print("Each (user, post) impression gets labels for ALL tasks:\n")
training_data = construct_multitask_training_data(interactions_df)

print(f"\nTotal training samples: {len(training_data)}")
print(f"\nSample rows:")
print(training_data.head(10).to_string(index=False))

print("\n=== Key Insight ===")
print("Notice how shares are much rarer than clicks.")
print("This is why multi-task learning helps -- the shared backbone")
print("transfers knowledge from data-rich tasks (clicks) to data-poor tasks (shares).")

## 4. Feature Engineering

Features are the **ingredients** that our model uses to make predictions. Think of it like deciding whether someone will enjoy a movie -- you would look at:
- What kind of movies they liked before (user features)
- What kind of movie it is (post features)
- Whether their best friend loved it (user-author affinity)

### The Three Feature Categories

```
              FEATURE CATEGORIES
    ========================================
    
    +------------------+
    | A. POST FEATURES |
    | - Text embedding |
    | - Image embedding|
    | - Reaction counts|
    | - Hashtags       |
    | - Post age       |
    +------------------+
    
    +------------------+
    | B. USER FEATURES |
    | - Demographics   |
    | - Device/time    |
    | - History        |
    | - @mentioned?    |
    +------------------+
    
    +-------------------------+
    | C. USER-AUTHOR AFFINITY |  <-- MOST IMPORTANT!
    | - Like/comment rate     |
    | - Friendship length     |
    | - Close friend flag     |
    +-------------------------+
```

In [ ]:
# === Feature Engineering Pipeline ===

def compute_post_features(post, current_time):
    """
    Compute features from the post itself.
    
    12-year-old version:
    Look at the post and ask: 'What kind of post is this?'
    Is it popular? Is it new? Does it have pictures?
    """
    # Post age (bucketized)
    age_hours = (current_time - post['created_at']).total_seconds() / 3600
    if age_hours < 1:
        age_bucket = 0  # Very fresh
    elif age_hours < 5:
        age_bucket = 1
    elif age_hours < 24:
        age_bucket = 2
    elif age_hours < 168:  # 7 days
        age_bucket = 3
    else:
        age_bucket = 4  # Old
    
    # Content type encoding
    type_map = {'text': 0, 'image': 1, 'video': 2, 'text+image': 3}
    content_type = type_map.get(post['post_type'], 0)
    
    return {
        'post_age_bucket': age_bucket,
        'content_type': content_type,
        'log_likes': np.log1p(post['num_likes']),
        'log_comments': np.log1p(post['num_comments']),
        'log_shares': np.log1p(post['num_shares']),
        'has_media': int(post['post_type'] in ['image', 'video', 'text+image']),
    }


def compute_user_features(user):
    """
    Compute features from the user's profile.
    """
    age = user['age']
    if age < 25:
        age_bucket = 0
    elif age < 35:
        age_bucket = 1
    elif age < 50:
        age_bucket = 2
    else:
        age_bucket = 3
    
    gender_map = {'M': 0, 'F': 1, 'Other': 2}
    country_map = {'US': 0, 'UK': 1, 'India': 2, 'Brazil': 3}
    
    return {
        'age_bucket': age_bucket,
        'gender': gender_map.get(user['gender'], 0),
        'country': country_map.get(user['country'], 0),
    }


def compute_affinity_features(user_id, author_id, interactions_df, friendships_df):
    """
    Compute user-author affinity features.
    
    12-year-old version:
    How much does this user like stuff from this author?
    Are they close friends? Do they always like each other's posts?
    
    Staff-level detail:
    According to Facebook research, user-author affinity is the
    SINGLE MOST PREDICTIVE feature for engagement.
    """
    # Get all interactions between this user and posts by this author
    author_posts = posts_df[posts_df['author_id'] == author_id]['post_id'].values
    user_author_interactions = interactions_df[
        (interactions_df['user_id'] == user_id) &
        (interactions_df['post_id'].isin(author_posts))
    ]
    
    total_impressions = len(user_author_interactions[user_author_interactions['interaction_type'] == 'impression'])
    total_impressions = max(total_impressions, 1)  # Avoid division by zero
    
    like_rate = len(user_author_interactions[user_author_interactions['interaction_type'] == 'like']) / total_impressions
    comment_rate = len(user_author_interactions[user_author_interactions['interaction_type'] == 'comment']) / total_impressions
    click_rate = len(user_author_interactions[user_author_interactions['interaction_type'] == 'click']) / total_impressions
    hide_rate = len(user_author_interactions[user_author_interactions['interaction_type'] == 'hide']) / total_impressions
    
    # Are they friends?
    pair = (min(user_id, author_id), max(user_id, author_id))
    friendship = friendships_df[
        (friendships_df['user_id_1'] == pair[0]) &
        (friendships_df['user_id_2'] == pair[1])
    ]
    is_friend = int(len(friendship) > 0)
    is_close_friend = int(len(friendship) > 0 and friendship.iloc[0]['close_friend'] == 1) if is_friend else 0
    
    return {
        'affinity_like_rate': round(like_rate, 4),
        'affinity_comment_rate': round(comment_rate, 4),
        'affinity_click_rate': round(click_rate, 4),
        'affinity_hide_rate': round(hide_rate, 4),
        'is_friend': is_friend,
        'is_close_friend': is_close_friend,
    }


# Demonstrate on one (user, post) pair
user = users_df.iloc[0]
post = posts_df.iloc[0]
current_time = datetime(2024, 3, 15, 14, 0)

post_feats = compute_post_features(post, current_time)
user_feats = compute_user_features(user)
affinity_feats = compute_affinity_features(user['user_id'], post['author_id'], interactions_df, friendships_df)

print(f"=== Features for User '{user['username']}' x Post '{post['text_content']}' ===")
print(f"\n--- Post Features ---")
for k, v in post_feats.items():
    print(f"  {k}: {v}")
print(f"\n--- User Features ---")
for k, v in user_feats.items():
    print(f"  {k}: {v}")
print(f"\n--- User-Author Affinity Features (MOST IMPORTANT) ---")
for k, v in affinity_feats.items():
    print(f"  {k}: {v}")

## 5. Multi-Task Model Architecture

### Why Multi-Task Learning?

Imagine you are training for a triathlon (swimming, biking, running). You COULD hire three separate coaches -- one for each sport. But a single smart coach who understands ALL three sports would be better because:
- Cardio training helps ALL three sports (shared knowledge)
- A coach who sees your biking weakness might notice it relates to your running form
- It is way cheaper to hire one coach than three

Multi-task learning is the same idea: one model with a **shared backbone** that learns patterns useful for ALL tasks, and **separate heads** that specialize in each task.

```
                    MULTI-TASK DNN ARCHITECTURE
    ============================================================
    
    Input Features (user + post + affinity)
              |
              v
    +--------------------+
    |   Shared Backbone  |  <-- Learns patterns useful for ALL tasks
    |   (Dense layers)   |      e.g., "close friends engage more"
    +----+----+----+-----+
         |    |    |    \\
         v    v    v     v
    +----+ +--+ +----+ +----+
    |Like| |Cm| |Shr | |Hide|  <-- Task-specific heads
    |Head| |Hd| |Head| |Head|      Each predicts one reaction
    +----+ +--+ +----+ +----+
      |     |     |      |
      v     v     v      v
    P(like) P(cm) P(shr) P(hide)
```

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# === Multi-Task News Feed Model ===

class MultiTaskNewsFeedModel(nn.Module):
    """
    Multi-task DNN for news feed ranking.
    
    12-year-old version:
    One big brain (shared backbone) that understands posts and users,
    with separate 'experts' (heads) for predicting each reaction type.
    
    Staff-level detail:
    - Shared backbone: learns cross-task representations
    - Task-specific heads: specialize per reaction type
    - Binary tasks (like, comment, share, hide): sigmoid output
    - Regression task (dwell time): ReLU output (non-negative)
    - Total loss = weighted sum of per-task losses
    """
    
    def __init__(self, input_dim, shared_dims=[256, 128], head_dims=[64, 32],
                 binary_tasks=None, regression_tasks=None):
        super().__init__()
        
        if binary_tasks is None:
            binary_tasks = ['click', 'like', 'comment', 'share', 'hide']
        if regression_tasks is None:
            regression_tasks = ['dwell_time']
        
        self.binary_tasks = binary_tasks
        self.regression_tasks = regression_tasks
        self.all_tasks = binary_tasks + regression_tasks
        
        # Shared backbone
        shared_layers = []
        prev_dim = input_dim
        for dim in shared_dims:
            shared_layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Dropout(0.3)
            ])
            prev_dim = dim
        self.shared_backbone = nn.Sequential(*shared_layers)
        
        # Task-specific heads
        self.task_heads = nn.ModuleDict()
        for task in self.all_tasks:
            head_layers = []
            head_prev_dim = shared_dims[-1]
            for dim in head_dims:
                head_layers.extend([
                    nn.Linear(head_prev_dim, dim),
                    nn.ReLU(),
                    nn.Dropout(0.2)
                ])
                head_prev_dim = dim
            head_layers.append(nn.Linear(head_prev_dim, 1))
            self.task_heads[task] = nn.Sequential(*head_layers)
    
    def forward(self, x):
        # Shared representation
        shared_repr = self.shared_backbone(x)
        
        # Task-specific outputs
        outputs = {}
        for task in self.binary_tasks:
            outputs[task] = torch.sigmoid(self.task_heads[task](shared_repr)).squeeze(-1)
        for task in self.regression_tasks:
            outputs[task] = torch.relu(self.task_heads[task](shared_repr)).squeeze(-1)
        
        return outputs


# Create model
input_dim = 15  # Number of features we engineered
model = MultiTaskNewsFeedModel(input_dim=input_dim)

print("=== Multi-Task News Feed Model ===")
print(f"\nInput dimension: {input_dim}")
print(f"Binary tasks: {model.binary_tasks}")
print(f"Regression tasks: {model.regression_tasks}")
print(f"\nArchitecture:")
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
shared_params = sum(p.numel() for p in model.shared_backbone.parameters())
head_params = total_params - shared_params
print(f"\nTotal parameters: {total_params:,}")
print(f"  Shared backbone: {shared_params:,} ({shared_params/total_params*100:.1f}%)")
print(f"  Task heads: {head_params:,} ({head_params/total_params*100:.1f}%)")
print(f"\nKey insight: Most parameters are SHARED, making this much more efficient")
print(f"than training {len(model.all_tasks)} separate models.")

In [ ]:
# === Training the Multi-Task Model ===

def generate_synthetic_news_feed_data(n_samples=2000, n_features=15):
    """
    Generate synthetic training data for the multi-task model.
    
    In production, this comes from the interaction logs we discussed above.
    Here we simulate realistic class imbalances:
    - Clicks: ~25% positive (most common reaction)
    - Likes: ~12% positive
    - Comments: ~6% positive
    - Shares: ~3% positive (rarest positive reaction)
    - Hides: ~2% positive
    - Dwell time: continuous, exponentially distributed
    """
    np.random.seed(42)
    X = np.random.randn(n_samples, n_features)
    
    # Create correlated labels (reactions are NOT independent)
    # A post you click is more likely to be one you also like
    base_signal = 0.3 * X[:, 0] + 0.2 * X[:, 5] + 0.4 * X[:, 10]  # affinity features matter most
    
    labels = {}
    # Binary tasks with different thresholds (more rare = higher threshold)
    labels['click'] = (base_signal + np.random.randn(n_samples) * 0.5 > 0.0).astype(float)
    labels['like'] = (base_signal + np.random.randn(n_samples) * 0.5 > 0.5).astype(float)
    labels['comment'] = (base_signal + np.random.randn(n_samples) * 0.5 > 1.0).astype(float)
    labels['share'] = (base_signal + np.random.randn(n_samples) * 0.5 > 1.5).astype(float)
    labels['hide'] = ((-base_signal) + np.random.randn(n_samples) * 0.5 > 1.5).astype(float)
    # Dwell time: positive correlation with engagement
    labels['dwell_time'] = np.maximum(0, 3.0 + 2.0 * base_signal + np.random.randn(n_samples) * 2.0)
    
    return X, labels


# Generate data
X, labels = generate_synthetic_news_feed_data(n_samples=3000, n_features=input_dim)

# Print class distribution
print("=== Training Data Class Distribution ===")
for task in ['click', 'like', 'comment', 'share', 'hide']:
    pos_rate = labels[task].mean() * 100
    print(f"  {task:<10}: {pos_rate:.1f}% positive")
print(f"  {'dwell_time':<10}: mean={labels['dwell_time'].mean():.1f}s, std={labels['dwell_time'].std():.1f}s")

# Split into train/test
split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
labels_train = {k: v[:split_idx] for k, v in labels.items()}
labels_test = {k: v[split_idx:] for k, v in labels.items()}

print(f"\nTrain: {len(X_train)} samples, Test: {len(X_test)} samples")

In [ ]:
from sklearn.metrics import roc_auc_score

# === Training Loop ===

def train_multitask_model(model, X_train, labels_train, X_test, labels_test,
                          n_epochs=30, batch_size=128, lr=0.001):
    """
    Train the multi-task model.
    
    The key idea: the total loss is a weighted sum of per-task losses.
    
    L_total = alpha_click * L_click + alpha_like * L_like + ...
    
    Binary tasks use BCE loss, regression tasks use Huber loss.
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)
    bce_loss_fn = nn.BCELoss()
    huber_loss_fn = nn.HuberLoss()
    
    # Task loss weights (how much each task contributes to total loss)
    task_loss_weights = {
        'click': 1.0,
        'like': 2.0,
        'comment': 3.0,
        'share': 4.0,
        'hide': 2.0,
        'dwell_time': 1.0,
    }
    
    # Prepare tensors
    X_train_t = torch.FloatTensor(X_train)
    X_test_t = torch.FloatTensor(X_test)
    labels_train_t = {k: torch.FloatTensor(v) for k, v in labels_train.items()}
    labels_test_t = {k: torch.FloatTensor(v) for k, v in labels_test.items()}
    
    dataset = TensorDataset(X_train_t, *[labels_train_t[t] for t in model.all_tasks])
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    history = {'epoch': [], 'total_loss': []}
    for task in model.binary_tasks:
        history[f'auc_{task}'] = []
    
    print("=== Training Multi-Task Model ===")
    print(f"Tasks: {model.all_tasks}")
    print(f"Loss weights: {task_loss_weights}")
    print()
    
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0
        
        for batch in loader:
            batch_X = batch[0]
            batch_labels = {task: batch[i+1] for i, task in enumerate(model.all_tasks)}
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            
            total_loss = 0.0
            for task in model.binary_tasks:
                task_loss = bce_loss_fn(outputs[task], batch_labels[task])
                total_loss += task_loss_weights[task] * task_loss
            for task in model.regression_tasks:
                task_loss = huber_loss_fn(outputs[task], batch_labels[task])
                total_loss += task_loss_weights[task] * task_loss
            
            total_loss.backward()
            optimizer.step()
            epoch_loss += total_loss.item()
        
        # Evaluate every 5 epochs
        if (epoch + 1) % 5 == 0:
            model.eval()
            with torch.no_grad():
                test_outputs = model(X_test_t)
            
            avg_loss = epoch_loss / len(loader)
            history['epoch'].append(epoch + 1)
            history['total_loss'].append(avg_loss)
            
            auc_str = ""
            for task in model.binary_tasks:
                try:
                    auc = roc_auc_score(labels_test[task], test_outputs[task].numpy())
                except ValueError:
                    auc = 0.5
                history[f'auc_{task}'].append(auc)
                auc_str += f"{task}={auc:.3f} "
            
            print(f"Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | AUC: {auc_str}")
    
    return history


history = train_multitask_model(model, X_train, labels_train, X_test, labels_test, n_epochs=30)

In [ ]:
# === Visualize Training Progress ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Loss curve
axes[0].plot(history['epoch'], history['total_loss'], 'b-o', linewidth=2, markersize=6)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Total Loss', fontsize=12)
axes[0].set_title('Training Loss (Multi-Task)', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Plot 2: AUC per task
colors = ['#2ecc71', '#3498db', '#9b59b6', '#e67e22', '#e74c3c']
for i, task in enumerate(model.binary_tasks):
    axes[1].plot(history['epoch'], history[f'auc_{task}'], '-o',
                 color=colors[i], linewidth=2, markersize=6, label=task)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('ROC-AUC', fontsize=12)
axes[1].set_title('Per-Task AUC on Test Set', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0.4, 1.0])

plt.tight_layout()
plt.show()

print("Key observations:")
print("  1. Loss decreases steadily -- the model is learning")
print("  2. Click AUC is typically highest -- it has the most training data")
print("  3. Share AUC may be lower -- sparse data makes it harder")
print("  4. But sharing the backbone HELPS sparse tasks vs training independently")

In [ ]:
# === Computing Engagement Scores and Ranking Posts ===

def rank_posts_for_user(model, user_features_batch, reaction_weights):
    """
    Given a batch of (user, post) feature vectors, compute engagement scores
    and return ranked post indices.
    
    12-year-old version:
    For each post, ask the model: 'How likely is this user to click, like,
    comment, share, or hide this post?' Multiply each by importance weights,
    add them up, and sort from highest to lowest.
    """
    model.eval()
    with torch.no_grad():
        outputs = model(torch.FloatTensor(user_features_batch))
    
    # Compute engagement score for each post
    n_posts = len(user_features_batch)
    scores = np.zeros(n_posts)
    
    for task in model.binary_tasks:
        weight = reaction_weights.get(task, 0)
        scores += outputs[task].numpy() * weight
    
    # Sort by engagement score (descending)
    ranked_indices = np.argsort(-scores)
    
    return scores, ranked_indices, outputs


# Simulate ranking 20 candidate posts for one user
np.random.seed(123)
n_candidates = 20
candidate_features = np.random.randn(n_candidates, input_dim)

engagement_weights = {'click': 1, 'like': 5, 'comment': 10, 'share': 20, 'hide': -10}

scores, ranked_indices, outputs = rank_posts_for_user(model, candidate_features, engagement_weights)

print("=== Ranked News Feed for User ===")
print(f"{'Rank':<6} {'Post':<8} {'Score':<10} {'P(click)':<10} {'P(like)':<10} {'P(cmt)':<10} {'P(share)':<10} {'P(hide)':<10}")
print("-" * 74)
for rank, idx in enumerate(ranked_indices[:10]):
    print(f"{rank+1:<6} Post_{idx:<4} {scores[idx]:<10.3f} "
          f"{outputs['click'][idx].item():<10.3f} "
          f"{outputs['like'][idx].item():<10.3f} "
          f"{outputs['comment'][idx].item():<10.3f} "
          f"{outputs['share'][idx].item():<10.3f} "
          f"{outputs['hide'][idx].item():<10.3f}")

print(f"\nShowing top 10 of {n_candidates} candidates.")
print("Posts with high share probability get a big score boost (weight=20).")
print("Posts the user would likely hide get penalized (weight=-10).")

## 6. Handling Passive Users

Here is a problem: many users scroll through their feed but **never** click, like, comment, or share anything. For these "lurkers," the multi-task model predicts near-zero probabilities for all reactions, making the engagement score almost zero for every post. That means we cannot meaningfully rank posts for them.

**Solution: Add implicit signal tasks.**

1. **Dwell-time prediction** (regression): How long will the user look at this post? Even lurkers spend more time on posts they find interesting.

2. **Skip prediction** (binary): Will the user spend less than 0.5 seconds on this post? A quick scroll past means the post was not engaging.

These signals work for ALL users, including the silent majority who never explicitly react.

In [ ]:
# === Passive Users: Dwell Time Matters ===

# Simulate dwell time distribution
np.random.seed(42)
n_impressions = 5000

# Active users: mix of short and long dwell times
active_dwell = np.concatenate([
    np.random.exponential(1.0, n_impressions // 2),   # Quick scrolls
    np.random.exponential(8.0, n_impressions // 2),   # Engaged reading
])

# Passive users: mostly short, but SOME posts get attention
passive_dwell = np.concatenate([
    np.random.exponential(0.5, int(n_impressions * 0.7)),  # Quick scrolls
    np.random.exponential(5.0, int(n_impressions * 0.3)),  # Occasional interest
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(np.clip(active_dwell, 0, 30), bins=50, color='#3498db', alpha=0.7, edgecolor='white')
axes[0].axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Skip threshold (0.5s)')
axes[0].set_xlabel('Dwell Time (seconds)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Active User Dwell Times', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)

axes[1].hist(np.clip(passive_dwell, 0, 30), bins=50, color='#e67e22', alpha=0.7, edgecolor='white')
axes[1].axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Skip threshold (0.5s)')
axes[1].set_xlabel('Dwell Time (seconds)', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Passive User Dwell Times', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

active_skip_rate = (active_dwell < 0.5).mean() * 100
passive_skip_rate = (passive_dwell < 0.5).mean() * 100

print(f"Active user skip rate: {active_skip_rate:.1f}%")
print(f"Passive user skip rate: {passive_skip_rate:.1f}%")
print(f"\nKey insight: Even passive users have DIFFERENT dwell times per post.")
print(f"Dwell time captures engagement that likes/comments miss entirely.")

## 7. Evaluation Metrics

### Offline vs Online Metrics

Think of it like testing a new recipe:
- **Offline metrics**: You taste the food yourself before serving it (quick, cheap, but YOU are not the customer)
- **Online metrics**: You serve it to actual customers and see if they come back (slow, expensive, but tells you the real truth)

In [ ]:
# === Evaluation Metrics ===

def evaluate_news_feed_model(model, X_test, labels_test):
    """
    Comprehensive offline evaluation of the multi-task model.
    """
    model.eval()
    with torch.no_grad():
        outputs = model(torch.FloatTensor(X_test))
    
    results = {}
    for task in model.binary_tasks:
        preds = outputs[task].numpy()
        true = labels_test[task]
        try:
            auc = roc_auc_score(true, preds)
        except ValueError:
            auc = 0.5
        results[task] = {'auc': auc, 'mean_pred': preds.mean(), 'pos_rate': true.mean()}
    
    # Dwell time evaluation
    dwell_preds = outputs['dwell_time'].numpy()
    dwell_true = labels_test['dwell_time']
    mae = np.mean(np.abs(dwell_preds - dwell_true))
    results['dwell_time'] = {'mae': mae, 'mean_pred': dwell_preds.mean(), 'mean_true': dwell_true.mean()}
    
    return results


results = evaluate_news_feed_model(model, X_test, labels_test)

print("=== Offline Evaluation Results ===")
print(f"\n{'Task':<12} {'ROC-AUC':<10} {'Avg Pred':<12} {'Actual Rate':<12}")
print("-" * 46)
for task in model.binary_tasks:
    r = results[task]
    print(f"{task:<12} {r['auc']:<10.4f} {r['mean_pred']:<12.4f} {r['pos_rate']:<12.4f}")

r = results['dwell_time']
print(f"\nDwell Time MAE: {r['mae']:.2f}s (predicted avg: {r['mean_pred']:.2f}s, actual avg: {r['mean_true']:.2f}s)")

print("\n=== Online Metrics (What to Measure in Production) ===")
online_metrics = {
    'CTR': 'clicked_posts / impressions -- basic engagement (but vulnerable to clickbait)',
    'Reaction Rates': 'liked_posts / impressions -- stronger signal than CTR',
    'Total Time Spent': 'aggregate dwell time over 1 week -- captures passive engagement',
    'User Satisfaction': 'explicit survey feedback -- most accurate but expensive',
}
for metric, desc in online_metrics.items():
    print(f"  {metric}: {desc}")

print("\nKey interview insight: CTR alone is INSUFFICIENT because clickbait inflates CTR.")
print("Always mention reaction rates and time spent as complementary metrics.")

## 8. Serving Architecture

The production system must deliver ranked posts in **under 200ms** for **2 billion daily active users**. This requires a carefully designed three-stage pipeline.

```
              SERVING ARCHITECTURE (< 200ms total)
    ============================================================
    
    User Opens Feed
          |
          v
    +-----+--------+
    | 1. RETRIEVAL  |  Fetch unseen posts from friends, groups, pages
    | (< 50ms)      |  Input: billions of posts -> Output: ~1000 candidates
    | Inverted index|  Uses social graph traversal + inverted indexes
    +-----+--------+
          |
          v
    +-----+--------+
    | 2. RANKING    |  Run multi-task DNN on each candidate
    | (< 100ms)     |  Compute weighted engagement score
    | Multi-task DNN|  Sort by score -> ranked list
    +-----+--------+
          |
          v
    +-----+--------+
    | 3. RE-RANKING |  Apply business logic on top of ML scores
    | (< 50ms)      |  - Diversity enforcement
    | Business rules|  - Sponsored content injection
    |               |  - Content policy filters
    +-----+--------+
          |
          v
    Personalized Feed --> User's Screen
```

In [ ]:
# === Simulating the Full Serving Pipeline ===

import time

class NewsFeedServingPipeline:
    """
    Simulates the three-stage serving pipeline.
    
    12-year-old version:
    Stage 1 (Retrieval): Get all new posts from your friends (like checking the mailbox)
    Stage 2 (Ranking): Score each post (like sorting mail by importance)
    Stage 3 (Re-ranking): Apply extra rules (like 'don't show 10 letters from the same person')
    """
    
    def __init__(self, model, reaction_weights):
        self.model = model
        self.reaction_weights = reaction_weights
    
    def stage1_retrieval(self, user_id, all_posts, friendships, n_candidates=100):
        """Fetch candidate posts for this user."""
        start = time.time()
        # In production: social graph traversal + inverted index
        # Here: simulate by randomly selecting candidates
        candidates = np.random.choice(len(all_posts), min(n_candidates, len(all_posts)), replace=False)
        latency = (time.time() - start) * 1000
        return candidates, latency
    
    def stage2_ranking(self, candidate_features):
        """Score and rank candidates using the multi-task model."""
        start = time.time()
        self.model.eval()
        with torch.no_grad():
            outputs = self.model(torch.FloatTensor(candidate_features))
        
        # Compute engagement scores
        scores = np.zeros(len(candidate_features))
        for task in self.model.binary_tasks:
            weight = self.reaction_weights.get(task, 0)
            scores += outputs[task].numpy() * weight
        
        ranked_indices = np.argsort(-scores)
        latency = (time.time() - start) * 1000
        return ranked_indices, scores, latency
    
    def stage3_reranking(self, ranked_indices, scores, post_metadata, top_k=20):
        """Apply business logic: diversity, sponsored content, content policy."""
        start = time.time()
        
        final_list = []
        seen_authors = {}
        max_per_author = 3  # Diversity: max 3 posts from same author
        
        for idx in ranked_indices:
            author = post_metadata[idx].get('author_id', 0)
            author_count = seen_authors.get(author, 0)
            
            if author_count < max_per_author:
                final_list.append(idx)
                seen_authors[author] = author_count + 1
            
            if len(final_list) >= top_k:
                break
        
        latency = (time.time() - start) * 1000
        return final_list, latency
    
    def serve(self, user_id, all_posts, friendships, post_metadata, top_k=20):
        """Full pipeline: retrieval -> ranking -> re-ranking."""
        # Stage 1
        candidates, lat1 = self.stage1_retrieval(user_id, all_posts, friendships)
        
        # Stage 2
        candidate_features = all_posts[candidates]
        ranked_indices, scores, lat2 = self.stage2_ranking(candidate_features)
        
        # Stage 3
        final_list, lat3 = self.stage3_reranking(ranked_indices, scores, 
                                                  [post_metadata[c] for c in candidates], top_k)
        
        total_latency = lat1 + lat2 + lat3
        return final_list, total_latency, (lat1, lat2, lat3)


# Simulate
np.random.seed(42)
n_all_posts = 500
all_post_features = np.random.randn(n_all_posts, input_dim)
post_metadata = [{'author_id': np.random.randint(1, 21), 'post_id': i} for i in range(n_all_posts)]

pipeline = NewsFeedServingPipeline(model, engagement_weights)
final_feed, total_lat, (lat1, lat2, lat3) = pipeline.serve(
    user_id=1, all_posts=all_post_features, friendships=None, post_metadata=post_metadata
)

print("=== Serving Pipeline Execution ===")
print(f"  Stage 1 (Retrieval):  {lat1:.2f}ms")
print(f"  Stage 2 (Ranking):    {lat2:.2f}ms")
print(f"  Stage 3 (Re-ranking): {lat3:.2f}ms")
print(f"  Total:                {total_lat:.2f}ms")
print(f"\n  Final feed: {len(final_feed)} posts")
print(f"  (In production, latency is dominated by network I/O and feature lookups,")
print(f"   not model inference. The model itself is very fast.)")

## 9. Quick Interview Cheat Sheet

```
1. CLARIFY:       What platform? What reactions? Latency? Scale?
2. ML OBJECTIVE:  Weighted engagement score (clicks*1 + likes*5 + comments*10 + shares*20)
3. DATA:          Users, Posts, Interactions, Friendship graph
4. FEATURES:      Post features + User features + User-Author affinity (MOST IMPORTANT!)
5. MODEL:         Multi-task DNN with shared backbone + N heads (one per reaction)
6. PASSIVE USERS: Add dwell-time (regression) and skip (binary) tasks
7. TRAINING:      Binary cross-entropy for classification, Huber for regression
8. OFFLINE:       ROC-AUC per task
9. ONLINE:        CTR, reaction rates, total time spent, user surveys
10. SERVING:      Retrieval -> Ranking -> Re-ranking (< 200ms total)
```

## Key Takeaways

1. **Multi-task learning is the key architecture** -- one shared backbone with N task-specific heads is more efficient and effective than N independent models

2. **User-author affinity is the most predictive feature** -- how a user has interacted with a specific author's past posts is the strongest signal

3. **Weighted engagement score** lets the business tune what matters (shares > likes > clicks, and hides/blocks are negative)

4. **Handle passive users** with implicit signals (dwell time, skip) that capture engagement even when users never explicitly react

5. **Three-stage serving** (retrieval -> ranking -> re-ranking) enables sub-200ms responses at billions-of-users scale

6. **CTR alone is insufficient** -- always mention reaction rates and time spent as complementary online metrics

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)